In [1]:
import os
import pandas as pd
import steamreviews
import time

In [2]:
games = {
    "This War of Mine": 282070,
    "Partisans 1941": 1227530,
    "Brothers in Arms Hell Highway": 15390,
    "Spec Ops The Line": 50300,
    "Metal Gear Solid V The Phantom Pain": 287700,
    "Ghost Recon Wildlands": 460930,
    "Call to Arms Ostfront": 400750,
    "Sniper Elite 5": 1029690
}

request_params = {
    "language": "english",
    "filter": "recent",
    "num_per_page": 100
}

In [3]:
save_dir = "data"
os.makedirs(save_dir, exist_ok=True)

In [5]:
TARGET_VALID = 10000

def download_filtered_reviews(app_id):
    cursor = "*"
    valid_reviews = []

    while True:
        batch, _ = steamreviews.download_reviews_for_app_id(
            app_id,
            chosen_request_params={**request_params, "cursor": cursor}
        )

        if not batch or "reviews" not in batch:
            break

        reviews = batch["reviews"]
        if not reviews:
            break

        for r in reviews.values():
            text = r.get("review", "")

            # ===== Filtering conditions =====
            if r.get("language") != "english":
                continue

            if len(text.split()) <= 30:
                continue

            author = r.get("author", {})

            row = {
                "review_text": text,
                "rating": r.get("voted_up"),
                "helpful": r.get("votes_up"),
                "funny": r.get("votes_funny"),

                # ===== Playtime variables =====
                "playtime_forever": author.get("playtime_forever", 0),
                "playtime_at_review": author.get("playtime_at_review", 0),
                "playtime_last_two_weeks": author.get("playtime_last_two_weeks", 0),

                "timestamp": r.get("timestamp_created")
            }

            valid_reviews.append(row)

            if len(valid_reviews) >= TARGET_VALID:
                return valid_reviews

        cursor = batch.get("cursor")

        if not cursor:
            break

        time.sleep(0.5)

    return valid_reviews

In [ ]:
def process_game(game_name, app_id):
    print(f"\n=== {game_name} ===")

    reviews = download_filtered_reviews(app_id)

    if len(reviews) < TARGET_VALID:
        print(f"Warning: Only got {len(reviews)} valid reviews")

    df = pd.DataFrame(reviews)

    # Convert time
    df["datetime"] = pd.to_datetime(df["timestamp"], unit="s", errors="coerce")
    df = df.sort_values(by="datetime", ascending=False)

    # Convert minutes → hours
    df["playtime_forever_hours"] = df["playtime_forever"] / 60
    df["playtime_at_review_hours"] = df["playtime_at_review"] / 60

    df = df.head(TARGET_VALID)

    path = os.path.join(save_dir, f"{app_id}.csv")

    df.to_csv(path, index=False, encoding="utf-8-sig")

    print(f"Saved {len(df)} reviews → {path}")


In [7]:
# ===== Batch run =====
for name, appid in games.items():
    process_game(name, appid)
    time.sleep(1)


=== This War of Mine ===
[appID = 282070] expected #reviews = 27224
Number of queries 150 reached. Cooldown: 310 seconds
Saved 10000 reviews → data\This_War_of_Mine_10000_filtered.csv

=== Partisans 1941 ===
[appID = 1227530] expected #reviews = 2024
Saved 1004 reviews → data\Partisans_1941_10000_filtered.csv

=== Brothers in Arms Hell Highway ===
[appID = 15390] expected #reviews = 1268
Saved 555 reviews → data\Brothers_in_Arms_Hell_Highway_10000_filtered.csv

=== Spec Ops The Line ===
[appID = 50300] expected #reviews = 27476
Number of queries 150 reached. Cooldown: 310 seconds
Saved 10000 reviews → data\Spec_Ops_The_Line_10000_filtered.csv

=== Metal Gear Solid V The Phantom Pain ===
[appID = 287700] expected #reviews = 59787
Number of queries 150 reached. Cooldown: 310 seconds
Number of queries 150 reached. Cooldown: 310 seconds
Number of queries 150 reached. Cooldown: 310 seconds
Saved 10000 reviews → data\Metal_Gear_Solid_V_The_Phantom_Pain_10000_filtered.csv

=== Ghost Recon Wi